In [19]:
#Buat SparkSession Baru
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv(
    "hdfs://localhost:9000/user/mazedev/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)
print("SparkSession dibuat")

SparkSession dibuat


In [20]:
from pyspark.sql import SparkSession

# Cek session yang aktif
active_spark = SparkSession.getActiveSession()

if active_spark:https://github.com/mazedev21/big-data/tree/main/pertemuan-4
    print("SparkSession sedang aktif!")
    # Anda bisa melihat konfigurasi atau detailnya, misalnya:
    print(f"Application Name: {active_spark.sparkContext.appName}")
else:
    print("Tidak ada SparkSession yang aktif.")

SparkSession sedang aktif!
Application Name: Tugas4


In [21]:
df.printSchema()
print("Jumlah Baris:", df.count())

print("Data 10 Baris Pertama:")
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah Baris: 1000
Data 10 Baris Pertama:
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan

In [27]:
# Menangani data kosong
from pyspark.sql.functions import col

nilai_null = df.filter(col("rating").isNull()).count()
print("Jumlah baris dengan Rating NULL: ", nilai_null)

#Fill data yang NULL
df_filled = df.na.fill({"rating": 3.0})

nilai_null = df_filled.filter(col("rating").isNull()).count()
print("Jumlah baris dengan Rating NULL: ", nilai_null)

Jumlah baris dengan Rating NULL:  204
Jumlah baris dengan Rating NULL:  0


In [30]:
#Transformasi Data
from pyspark.sql.functions import col, when
df_added = df_filled.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

df_added = df_added.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_added.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [34]:
# Analisis dengan group by
from pyspark.sql.functions import col, sum, count, avg, desc

# Kategori Tertinggi
kategori_tertinggi = df_added \
    .groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("pendapatan_kategori")) \
    .sort(desc("pendapatan_kategori"))

print("--- 1. Kategori dengan Total Pendapatan Tertinggi ---")
kategori_tertinggi.show(1)

# Kota dengan Tier Besar
kota_tier_besar = df_added \
    .filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("tier_transaksi").alias("jumlah_transaksi_besar")) \
    .sort(desc("jumlah_transaksi_besar"))

print("--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---")
kota_tier_besar.show(1)

#Rata-rata rating untuk masing-masing
rata_rating_pembayaran = df_added \
    .groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc())

print("--- 3. Rata-rata Rating per Metode Pembayaran ---")
rata_rating_pembayaran.show()

--- 1. Kategori dengan Total Pendapatan Tertinggi ---
+------------+-------------------+
|    kategori|pendapatan_kategori|
+------------+-------------------+
|Rumah Tangga|          138665000|
+------------+-------------------+
only showing top 1 row

--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

--- 3. Rata-rata Rating per Metode Pembayaran ---
+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|3.948207171314741|
|    Transfer Bank|3.932806324110672|
|         E-Wallet|            3.904|
|     Kartu Kredit|3.861788617886179|
+-----------------+-----------------+



In [35]:
# Menyimpan ke hdfs
output_dir = "hdfs://localhost:9000/user/mazedev/tugas4/proccessed_transaksi_september_2026"

df_added.write.csv(output_dir, header=True, mode="overwrite")
print("Berkas berhasil disimpan di HDFS")

!hdfs dfs -ls /user/mazedev/tugas4

Berkas berhasil disimpan di HDFS
Found 2 items
drwxr-xr-x   - mazedev supergroup          0 2026-09-12 22:43 /user/mazedev/tugas4/proccessed_transaksi_september_2026
-rw-r--r--   1 mazedev supergroup      83497 2026-09-12 21:29 /user/mazedev/tugas4/transaksi_september_2026.csv


In [36]:
# Menutup session
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
